## Silver Layer: fact_orders — 基础订单宽表

**职责**: 将 Bronze 层核心事实表 JOIN 为一张订单宽表，供 Gold 层使用。

**设计原则**:
- Silver 层只做基础 JOIN 和清洗，不做 9 表完整 JOIN（完整宽表由 dbt 的 `fact_orders.sql` 负责）
- 与 dbt 的职责边界：Spark 处理格式转换和基础清洗，dbt 做业务建模和指标定义

**输入**: bronze_orders / bronze_order_items / bronze_customers
**输出**: `silver_fact_orders` Delta 表

In [0]:
from pyspark.sql import functions as F

In [0]:
# 从 Bronze 层读取 3 张核心表
print("正在读取铜牌层物理资产...")
oi = spark.table("bronze_order_items")
print("bronze_order_items OK...")
o = spark.table("bronze_orders")
print("bronze_orders OK...")
c = spark.table("bronze_customers")
print("bronze_customers OK...")

In [0]:
# 3 表 JOIN + 基础清洗
# 注意：Silver 层只做基础 JOIN，不做 9 表完整 JOIN
# 完整宽表（含 payments/reviews/sellers/products/geo/translation）由 dbt fact_orders 模型负责
silver_fact_orders = oi \
    .join(o, on='order_id', how="inner") \
    .join(c, on='customer_id', how="inner") \
    .filter(F.col("price").isNotNull()) \
    .withColumn("price", F.round(F.col("price"), 2))

In [0]:
# 物理落盘成银牌资产表
silver_fact_orders.write.format("delta").mode("overwrite").saveAsTable("silver_fact_orders")
print(f"✅ 银牌层核心事实表 silver_fact_orders 物理落盘成功！当前明细行数: {silver_fact_orders.count():,}")